# pi-eval on Kaggle (2 x T4)

This notebook serves a model locally with an OpenAI-compatible endpoint, then runs `pi-eval` against it with the **default Pi profile and default tool spec**.

Kaggle settings before running:
- Accelerator: **2 x T4**
- Internet: **On** (needed for Node, vLLM, model download, and the repo clone)
- Add keys as Kaggle Secrets only if your repo/model needs them.

The task prompts contain **no tool or library hints**. The evaluated agent must discover what is available in its environment and install/use what it needs. This notebook deliberately does **not** install PDF/OCR/office extraction libraries into the agent environment.

In [ ]:
import os, time, subprocess, json
from pathlib import Path

REPO_URL = os.environ.get('PI_EVAL_REPO_URL', '')  # e.g. https://github.com/you/pi-eval.git
REPO_DIR = Path('/kaggle/working/pi-eval')
MODEL_ID = 'Qwen/Qwen3-4B'
SERVED_MODEL = 'qwen3-4b'
SUITE = 'extraction'
RUNS = 5
PORT = 8000
API_BASE = f'http://127.0.0.1:{PORT}/v1'
OUTPUT = Path('/kaggle/working/pi-eval-run')
print({'model': MODEL_ID, 'suite': SUITE, 'runs': RUNS, 'api': API_BASE})

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h
!nproc

## 1. Install Node 22+ (the harness needs Node 22.19 or newer)

In [ ]:
!cd /tmp && wget -q https://nodejs.org/dist/v22.20.0/node-v22.20.0-linux-x64.tar.xz && tar -xf node-v22.20.0-linux-x64.tar.xz && cp -r node-v22.20.0-linux-x64/* /usr/local/
!node --version && npm --version

## 2. Get the harness

Set `REPO_URL` in the config cell, or upload the repository as a Kaggle Dataset and adjust the copy command.

In [ ]:
if not REPO_URL:
    raise ValueError('Set REPO_URL to your pi-eval repository URL (or copy the repo into /kaggle/working/pi-eval).')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
print(sorted(p.name for p in REPO_DIR.iterdir())[:12])

## 3. Serve the model locally

Qwen3-4B is used because stable vLLM support on T4 is reliable. YaRN extends its 32k native context to 131k, which fits in 2 x 16 GB with tensor parallelism.

If vLLM fails on Turing (SM75), try `VLLM_ATTENTION_BACKEND=XFORMERS` or switch the model/serving stack. This is infrastructure, not part of the evaluated agent capability.

In [ ]:
!pip -q install vllm huggingface_hub
import subprocess, os, time
serve = (
    'nohup python -m vllm.entrypoints.openai.api_server '
    f'--model {MODEL_ID} --served-model-name {SERVED_MODEL} '
    '--tensor-parallel-size 2 --max-model-len 131072 '
    '--rope-scaling \'{"rope_type":"yarn","factor":4.0,"original_max_position_embeddings":32768}\' '
    '--enable-auto-tool-choice --tool-call-parser hermes '
    f'    --port {PORT} > /kaggle/working/vllm.log 2>&1 &'
)
subprocess.run(serve, shell=True, check=True)
print('vLLM starting; log: /kaggle/working/vllm.log')

In [ ]:
import requests, time
for attempt in range(120):
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            print('ready:', [m['id'] for m in response.json().get('data', [])])
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError('vLLM did not become ready. Check /kaggle/working/vllm.log')
!tail -n 20 /kaggle/working/vllm.log

## 4. Run the eval

Only the notebook client dependencies are installed here. Do not preinstall document/OCR libraries: the agent must discover and install what it needs.

In [ ]:
!pip -q install requests pandas
import sys
sys.path.insert(0, str(REPO_DIR / 'packages' / 'evals' / 'notebooks'))
from pi_eval import PiEval

client = PiEval(repo_path=REPO_DIR, api_base=API_BASE, api_key='EMPTY', timeout=7200)
print(client.health())

In [ ]:
result = client.run(
    suite=SUITE,
    backend='host',
    runs=RUNS,
    model=SERVED_MODEL,
    api_base=API_BASE,
    api_key='EMPTY',
)
print('ok:', result.ok)
print(result.report)

In [ ]:
result.save(OUTPUT)
print('saved:', sorted(p.name for p in OUTPUT.iterdir()))
result.traces_frame().head(20)

## 5. Cleanup

In [ ]:
!pkill -f 'vllm.entrypoints.openai.api_server' || true